In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

In [2]:
# ==========================================
# Your existing data
# X 4D (20,4)
# y 1D (20,)
# ==========================================

# Example
X = np.load('function_5/initial_inputs.npy')
y = np.load('function_5/initial_outputs.npy')

X = np.vstack([X, [0.121513, 0.471871, 0.879484, 0.957640],
                  [0.813595, 0.577155, 0.879484, 0.889234],
                  [0.836478, 0.044486, 0.879484, 0.898853],
                  [0.836478, 0.859340, 0.879484, 0.880891],
                  [0.836478, 0.862540, 0.879484, 0.957644],
                  [0.119879, 0.038193, 0.088947, 0.957644],
                  [0.836478, 0.862540, 0.879484, 0.957644],
                  [0.119879, 0.038193, 0.088947, 0.072880],
                  [0.833630, 0.836286, 0.099708, 0.949210],
                  [0.832072, 0.093453, 0.233589, 0.956109],
                  [0.748223, 0.044982, 0.819430, 0.110320]])

y = np.append(y, [728.9077561034792, 
                  1211.588143127307,
                  1140.9996504044836,
                  2212.888697754458,
                  2860.6448644329735,
                  93.72428641997294,
                  2860.6448644329735,
                  162.46984595179353,
                  1217.86676448125,
                  502.6890419604954,
                  62.34681940011192])

In [4]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=np.ones(X.shape[1]), nu=2.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# ==========================================
# Expected Improvement
# ==========================================

def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)

    mu, sigma = gp.predict(x, return_std=True)

    sigma = sigma[0]
    mu = mu[0]

    if sigma < 1e-12:
        return 0.0

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei



# ==========================================
# Optimise acquisition function
# ==========================================

bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits

y_best = np.max(y)


def objective(x):
    return -expected_improvement(x, gp, y_best)


best_x = None
best_ei = -np.inf

for _ in range(5000):

    x0 = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds]
    )

    res = minimize(
        objective,
        x0=x0,
        bounds=bounds,
        method="L-BFGS-B"
    )

    ei = -res.fun

    if ei > best_ei:
        best_ei = ei
        best_x = res.x

print("Next point to evaluate:")
next_point = np.round(best_x, decimals=6)
print(next_point)

print("Expected Improvement:")
print(best_ei)

C:\Users\mho2\AppData\Roaming\Python\Python310\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Next point to evaluate:
[0.209324 0.045862 0.869799 0.889402]
Expected Improvement:
2.8349421459190143e-11
